# How Louvain and Otsu Work in Stage 5

---

## Setup

After Stages 1-4, we have 60 SNOMED CUIs. These are a mix of diseases, drugs, findings, procedures. Many are near-duplicates — same concept, different SNOMED code.

Our job: remove the duplicates, keep the distinct ones.

---

## Step 1: Embeddings → Similarity Matrix

Each CUI has a 3072-dim embedding from BigQuery. We compute cosine similarity for all 60×60 pairs.

Cosine distance = 1 − similarity. Small distance = similar. Large = different.

This gives us a 60×60 matrix — 1,770 unique pairwise distances. Some are tiny (0.02 — near duplicates), some are huge (0.85 — completely unrelated). This matrix is used to **find** the k nearest neighbors, but the edge threshold comes from a different calculation (next step).

---

## Step 2: k-NN Graph (k=20)

For each CUI, find its 20 most similar CUIs from the 60×60 matrix. Then record the distance to the **20th neighbor** (the weakest connection in that CUI's neighborhood). This gives us **60 numbers** — one per CUI.

```
For CUI #1 (Severe asthma):
    find 20 nearest neighbors from the 60×60 matrix
    neighbors: Acute severe asthma (dist=0.01), Mild persistent (dist=0.08), ...
    20th neighbor: Asthma monitoring (dist=0.25)
    record → 0.25

For CUI #2 (Albuterol):
    find 20 nearest neighbors
    20th neighbor: Prednisone (dist=0.30)
    record → 0.30

For CUI #3 (Chest X-ray):
    find 20 nearest neighbors
    20th neighbor: Patient education (dist=0.48)
    record → 0.48

For CUI #4 (Follow-up visit):
    find 20 nearest neighbors
    20th neighbor: Hospitalization (dist=0.52)
    record → 0.52

...repeat for all 60 CUIs...
```

Now we have 60 numbers. Each number answers: "how far does this CUI have to reach to find its 20th neighbor?"

- Dense neighborhood (asthma): 0.25 — many similar CUIs nearby
- Sparse neighborhood (encounters): 0.52 — has to reach far to get 20 neighbors

**Edge threshold = mean of these 60 numbers ≈ 0.38**

NOT the mean of 1,770 pairwise distances. Just 60 values — one per CUI.

```
threshold = mean(0.25, 0.30, 0.48, 0.52, ...) across all 60 CUIs ≈ 0.38
```

**Building the graph:** for each CUI's 20 neighbors, add an edge only if that neighbor's distance < 0.38. Edge weight = cosine similarity.

Result: a sparse graph. Asthma CUIs connect to asthma CUIs (distances 0.01–0.15, all below 0.38). Asthma rarely connects to medications (distances 0.40–0.60, mostly above 0.38).

---

## Step 3: How Louvain Works

Louvain gets the graph. It doesn't know what the CUIs mean. It only sees nodes and weighted edges.

**Goal:** find groups of nodes that are more connected to each other than to the rest.

**Algorithm:**

```
Start: every CUI is its own community (60 communities of 1)

Phase 1 — move nodes:
  Pick CUI "Severe asthma". Try moving it to each neighbor's community.
  Moving to "Acute severe asthma"'s community improves modularity by +0.12
  Moving to "Albuterol"'s community improves by +0.01
  Best move: join "Acute severe asthma" → do it.
  
  Repeat for all 60 CUIs. Keep going until no move improves modularity.

Phase 2 — collapse:
  Each community becomes a single super-node.
  60 CUIs → maybe 8 super-nodes.
  Build a new graph of super-nodes.
  Run Phase 1 again on this smaller graph.

Repeat until stable.
```

**Result for our 60 CUIs:**

```
community 0 (10 CUIs): Severe persistent asthma, Acute severe asthma,
                        Severe asthma attack, Mild persistent + exacerb, ...
                        → Louvain grouped asthma severity CUIs together

community 1 (10 CUIs): Cough, Productive cough, Dry cough, Worsening cough,
                        Aggravated cough, Wheezing, Chest tightness, ...
                        → respiratory findings together

community 2 (10 CUIs): Albuterol, Albuterol sulfate, Fluticasone,
                        Montelukast, Prednisone, ...
                        → medications together

community 3 (10 CUIs): FEV, FEV1/FVC, Decreased FEV, Respiratory rate, ...
                        → lab values together

community 4 (10 CUIs): Spirometry, Lung auscultation, Nebulizer, ...
                        → procedures together

community 5 (10 CUIs): Follow-up visit, Physical exam, Patient education, ...
                        → encounters together
```

**All 60 CUIs are still here.** Louvain doesn't remove anything. It just groups them. The duplicates (Severe asthma attack ≈ Acute severe asthma) are now in the same community, ready for dedup.

---

## Step 4: How Otsu Works (within each community)

Louvain put 10 asthma CUIs in community 0. They're all related — that's why they're in the same community. But some are **duplicates** (same concept, different code) and some are **genuinely different** (mild vs severe). Otsu's job is to find the distance boundary between these two.

**Start with the 45 pairwise distances inside community 0:**

Every pair of CUIs in the community has a cosine distance. 10 CUIs = 45 pairs. Sort them:

```
pair                                              distance    what it really is
─────────────────────────────────────────────────────────────────────────────────
Severe asthma attack  ↔  Acute severe asthma      0.008       same concept, different code
Mild persistent uncntrl ↔ Mild persistent+rhinitis 0.012       same concept, different code
(pair 3)                                           0.015       same concept, different code
(pair 4)                                           0.018       same concept, different code

                        ─── natural gap ───

Mild persistent  ↔  Moderate persistent            0.095       different severity — NOT duplicate
Moderate ↔ Severe persistent                       0.110       different severity — NOT duplicate
(pair 7)                                           0.125       different concepts
(pair 8)                                           0.140       different concepts
...41 more pairs up to 0.22...
```

See the gap? Pairs 1-4 (distances 0.008–0.018) are duplicates. Pairs 5+ (distances 0.095+) are different concepts. There's a clear jump from 0.018 to 0.095.

**Otsu finds this gap automatically.** Here's how:

It walks through the sorted distances and tries each one as a potential dividing line. For each candidate:

- Split the 45 distances into "below" and "above" the candidate
- Compute the mean of each side
- Score = how different those two means are (weighted by group sizes)

The candidate that makes the two sides **most different** wins.

```
candidate = 0.012:
  below: {0.008}                       mean = 0.008
  above: {0.012, 0.015, ... 0.22}      mean = 0.128
  score = (1/45) × (44/45) × (0.008 - 0.128)² = 0.0003  (low — only 1 value below)

candidate = 0.095:                                          ← THE GAP
  below: {0.008, 0.012, 0.015, 0.018}  mean = 0.013       (the duplicates)
  above: {0.095, 0.110, ... 0.22}      mean = 0.145       (the different concepts)
  score = (4/45) × (41/45) × (0.013 - 0.145)² = 0.00143  (HIGHEST — clean split)

candidate = 0.110:
  below: {0.008...0.095}               mean = 0.047
  above: {0.110, ... 0.22}             mean = 0.158
  score = 0.00098  (worse — this split mixes duplicates with non-duplicates below)
```

**Winner: threshold = 0.095**

Safety check: cap at median (0.12). Since 0.095 < 0.12, no cap needed.

**What happens with this threshold:**

```
Otsu threshold = 0.095

4 pairs with distance < 0.095  → these are CANDIDATE duplicates
                                  (still need to pass guards before removal)

41 pairs with distance ≥ 0.095 → these are definitely different concepts
                                  (not even checked — skip them)
```

Otsu does NOT remove anything. It just draws a line:
- Below the line: "these pairs are suspicious — check them with the guards"
- Above the line: "these pairs are fine — don't bother checking"

The guards decide what actually gets removed (Step 5).

**Why each community gets its own Otsu threshold:**

```
community 0 (asthma):      threshold = 0.095  (severity variants are moderately spread)
community 2 (medications): threshold = 0.050  (drug names cluster very tightly)
community 5 (encounters):  threshold = 0.120  (encounter types are more spread out)
```

If we ran Otsu on all 1,770 distances (before Louvain), the threshold would be ~0.30 — too high for medications (would miss drug duplicates at 0.04) and too low for encounters (would flag non-duplicates at 0.11). Louvain scoping lets each community get the right threshold for its data.

---

## Step 5: Guards Check Each Candidate

Being below the Otsu threshold doesn't mean removal. Guards protect legitimate pairs.

**Example 1 — true duplicate (removed):**
```
C0264429 (Severe asthma attack)
C0264423 (Acute severe asthma)
dist = 0.008

guard 1: parent-child?      NO
guard 2: types disjoint?    NO (both Disease)
guard 3: ancestors disjoint? NO (both under Persistent asthma)
guard 4: sibling leaves?    NO

→ all failed → REMOVE C0264429 (lower IC)
```

**Example 2 — protected by guard 1 (kept):**
```
C0004096 (Asthma)                    IC=11.2
C0264408 (Severe persistent asthma)  IC=13.8
dist = 0.03

guard 1: parent-child?  YES — Asthma is parent of Severe persistent
→ STOP. KEEP BOTH.
```

**Example 3 — protected by guard 2 (if it happened):**
```
C0001927 (Albuterol)    [Drug]
C0199176 (Spirometry)   [Procedure]
dist = 0.04  (hypothetically close in embeddings)

guard 2: types disjoint?  YES — Drug ≠ Procedure
→ STOP. KEEP BOTH.
```
(This pair wouldn't appear because Louvain puts them in different communities. But if it did, guard 2 catches it.)

---

## Step 6: Repeat for every community

Each community gets its **own** Otsu threshold because each has different internal distances.

```
community 0 (asthma):      10 → 8   tight_t=0.095  (2 severity duplicates removed)
community 1 (findings):    10 → 9   tight_t=0.080  (aggravated cough ≈ worsening cough)
community 2 (medications): 10 → 6   tight_t=0.050  (4 salt/dose duplicates removed)
community 3 (labs):        10 → 9   tight_t=0.070  (decreased FEV ≈ decreased FEV1)
community 4 (procedures):  10 → 9   tight_t=0.085  (PFT ≈ spirometry)
community 5 (encounters):  10 → 7   tight_t=0.120  (3 follow-up duplicates removed)
──────────────────────────────────────
total:                     60 → 48   (12 true duplicates removed)
```

Note how medications get tight_t=0.050 (drug names are very similar in embedding space) while encounters get 0.120 (more spread out). Otsu adapts to each community's data.

---

## What Goes Where

```
60 CUIs after hierarchy
    │
    ├── embeddings fetched from BigQuery (60 × 3072-dim)
    │
    ├── 60×60 cosine similarity matrix (used to FIND k neighbors)
    │
    ├── k-NN: for each CUI, record 20th-neighbor distance → 60 values
    │         edge threshold = mean of these 60 values ≈ 0.38
    │
    ├── build graph: edges where neighbor distance < 0.38
    │
    ├── Louvain → 6 communities (groups similar CUIs, removes nothing)
    │
    ├── within each community:
    │       its own pairwise distances (e.g. 10×10 → 45 pairs)
    │       its own Otsu threshold (from those 45 distances)
    │       pairs below threshold → 4 guards check
    │       all guards fail → remove lower-IC duplicate
    │
    ├── ~48 surviving CUIs
    │
    └── → Stage 6: HierarchicalTopicBuilder
          (forms clinical topics from UMLS ancestry)
          (louvain communities thrown away — they were just for dedup scoping)
```

---

## Why Louvain Before Otsu?

Without Louvain, Otsu runs on all 1,770 pairwise distances. The distribution is bimodal — within-category pairs are tight (0.01–0.15) and cross-category pairs are far (0.40–0.85). Otsu would find a threshold around 0.30 which is too high — it'd miss tight duplicates within medications (should be 0.05).

With Louvain, each community gets its own Otsu threshold tuned to its own distance distribution. Medications get 0.05, asthma gets 0.095, encounters get 0.12. Each threshold is right for its data.

That's it. Louvain scopes. Otsu finds the threshold. Guards protect.
---

## Why Not Just Use Embedding Distance Directly?

The obvious question: we have embeddings, we can compute cosine distance between every pair. Why not just set a threshold (say 0.10) and remove everything below it? Why do we need Louvain at all?

**Problem 1: No single threshold works.**

```
drug pairs:       albuterol ↔ albuterol sulfate     dist = 0.03  (duplicate)
drug pairs:       albuterol ↔ montelukast            dist = 0.08  (different drug!)

encounter pairs:  follow-up visit ↔ medical follow-up dist = 0.09  (duplicate)
encounter pairs:  follow-up visit ↔ hospitalization   dist = 0.15  (different!)
```

If you set threshold = 0.10:
- Catches the drug duplicate (0.03 < 0.10) ✓
- But also flags albuterol vs montelukast (0.08 < 0.10) as duplicate ✗ — they're different drugs!
- Misses the encounter duplicate (0.09 < 0.10 barely catches it, but 0.15 is safe) — fragile

Drugs are tightly clustered in embedding space (all drug names look similar). Encounters are spread out ("follow-up" and "hospitalization" are very different). One threshold can't serve both.

Louvain separates drugs from encounters. Then each group gets its own Otsu threshold: 0.05 for drugs, 0.12 for encounters. Each threshold is right for its data.

**Problem 2: Comparing everything against everything is noisy.**

60 CUIs = 1,770 pairs. Most are cross-category (asthma vs drug, lab vs procedure). These are obviously not duplicates but they flood the distance distribution:

```
without Louvain — all 1,770 distances:
  0.03, 0.04, 0.08, 0.09, 0.12, ...  (within-category: some dups, some not)
  0.40, 0.45, 0.50, 0.55, 0.60, ...  (cross-category: obviously different)
  0.70, 0.75, 0.80, 0.85, ...         (totally unrelated)

Otsu on this → threshold ≈ 0.30 (splits within-category from cross-category)
That's useless — 0.30 is way above the real duplicate boundary (0.05–0.12)
```

The cross-category distances dominate and push Otsu's threshold too high. The duplicates (0.03–0.09) are invisible in the noise.

```
with Louvain — community 2 (medications only), 45 distances:
  0.03, 0.04, 0.05, ...  (duplicates: salt/dose variants)
  0.08, 0.10, 0.12, ...  (different drugs)

Otsu on this → threshold ≈ 0.06 (splits duplicates from different drugs)
That's exactly right.
```

Louvain removes the cross-category noise so Otsu can see the real signal within each group.

**Problem 3: Guards need context.**

The 4 structural guards check hierarchy edges, semantic types, ancestor sets, and sibling status. These checks are O(n) per pair (BFS for ancestors). Running them on all 1,770 pairs = expensive and mostly wasted on cross-category pairs that are obviously different.

With Louvain scoping: Otsu narrows to ~4-8 candidate pairs per community. Guards run on ~30 pairs total instead of 1,770. Same result, 50× fewer guard checks.

**Summary:**

```
just embeddings + fixed threshold:  one threshold for all → wrong for most categories
just embeddings + global Otsu:      cross-category noise drowns the real signal
Louvain + per-community Otsu:       each category gets the right threshold
                                    guards only run on real candidates
                                    50× fewer checks, better accuracy
```
